In [1]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/Reyezcode511/"
    "healthcare-data-engineering-pipeline/main/"
    "data/clean_healthcare_data.csv"
)

urllib.request.urlretrieve(url, "clean_healthcare_data.csv")

print("Healthcare data downloaded successfully.")

Healthcare data downloaded successfully.


In [2]:
%pip install -q pyspark

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("HealthcareSparkPipeline")
    .getOrCreate()
)

healthcare_df = spark.read.csv(
    "clean_healthcare_data.csv",
    header=True,
    inferSchema=True,
)

print("Spark version:", spark.version)
print("Records loaded:", healthcare_df.count())

healthcare_df.printSchema()
healthcare_df.show()

Spark version: 4.0.4
Records loaded: 7
root
 |-- patient_id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- hospital: string (nullable = true)
 |-- treatment_cost: double (nullable = true)

+----------+----+------+------------+----------------+--------------+
|patient_id| age|gender|   diagnosis|        hospital|treatment_cost|
+----------+----+------+------------+----------------+--------------+
|      1001|45.0|  Male|    Diabetes|Central Hospital|        1500.0|
|      1002|32.0|Female|Hypertension|   City Hospital|        1200.0|
|      1003|67.0|  Male|    Diabetes|Central Hospital|        2100.0|
|      1005|43.0|  Male|Hypertension|   City Hospital|        1450.0|
|      1006|41.0|Female|      Asthma|General Hospital|         950.0|
|      1007|73.0|  Male|    Diabetes|Central Hospital|        2500.0|
|      1008|38.0|Female|Hypertension|   City Hospital|        1400.0|
+----------+

In [4]:
from pyspark.sql import functions as F

diagnosis_summary = (
    healthcare_df.groupBy("diagnosis")
    .agg(
        F.sum("treatment_cost").alias("total_cost"),
        F.avg("treatment_cost").alias("average_cost"),
        F.count("*").alias("record_count"),
    )
    .orderBy(F.desc("total_cost"))
)

hospital_summary = (
    healthcare_df.groupBy("hospital")
    .agg(
        F.sum("treatment_cost").alias("total_cost"),
        F.avg("treatment_cost").alias("average_cost"),
        F.count("*").alias("record_count"),
    )
    .orderBy(F.desc("total_cost"))
)

print("SPARK DIAGNOSIS SUMMARY")
diagnosis_summary.show(truncate=False)

print("SPARK HOSPITAL SUMMARY")
hospital_summary.show(truncate=False)

total_cost = healthcare_df.agg(
    F.sum("treatment_cost").alias("total")
).first()["total"]

assert healthcare_df.count() == 7, "Expected 7 records."
assert abs(total_cost - 11100.0) < 0.01, "Unexpected total cost."

print("Checks passed: 7 records and total treatment cost of 11,100.")

SPARK DIAGNOSIS SUMMARY
+------------+----------+------------------+------------+
|diagnosis   |total_cost|average_cost      |record_count|
+------------+----------+------------------+------------+
|Diabetes    |6100.0    |2033.3333333333333|3           |
|Hypertension|4050.0    |1350.0            |3           |
|Asthma      |950.0     |950.0             |1           |
+------------+----------+------------------+------------+

SPARK HOSPITAL SUMMARY
+----------------+----------+------------------+------------+
|hospital        |total_cost|average_cost      |record_count|
+----------------+----------+------------------+------------+
|Central Hospital|6100.0    |2033.3333333333333|3           |
|City Hospital   |4050.0    |1350.0            |3           |
|General Hospital|950.0     |950.0             |1           |
+----------------+----------+------------------+------------+

Checks passed: 7 records and total treatment cost of 11,100.
